# CANVAS-CTX: habitat inference with spatial context

Predicting tissue habitats from histology, and testing whether giving the
classifier its spatial neighbourhood helps.

## Read the result before running it

The benchmark has already been run to completion on six seeds and four modes.
**The answer is close to no.** Adding graph context gives about +0.011 macro-F1
against a between-seed standard deviation of 0.046, and `grid3d`, the most
expensive mode, gives +0.0005 with three wins in six seeds. On Cohen's kappa
the graph mode does win on all six seeds (p = 0.031), which is the strongest
claim available and still a small one.

**Every run collapsed at least one habitat class**, median two of ten, and one
class was never predicted in 21 of 24 runs despite holding a normal share of
the data. A macro-F1 comparison between models that both fail to predict two
classes is not a reliable comparison, which is the first thing to fix.

This notebook exists so the benchmark can be re-run and that conclusion checked
or overturned, not because the result is settled in its favour.

## Before you start

**Runtime, Change runtime type, T4 GPU, Save.** The benchmark trains 24 models
and is much faster on a GPU, though it will run on CPU.

In [ ]:
#@title 1. Environment and code { display-mode: "form" }
import subprocess, sys, os, shutil
from pathlib import Path

REPO = 'https://github.com/swatian1989/canvas-ctx.git'
WORK = Path('/content/canvas')
if not WORK.exists():
    subprocess.run(['git', 'clone', '-q', REPO, str(WORK)], check=True)
os.chdir(WORK)
sys.path.insert(0, str(WORK / 'src'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'scikit-learn', 'scikit-image', 'pandas', 'pyarrow',
                'matplotlib', 'tifffile', 'pyyaml'], check=False)

import torch
print('working dir:', os.getcwd())
print('GPU        :', torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else 'none, the benchmark will be slow')
print('free disk  : %.0f GB' % (shutil.disk_usage('/content').free / 1e9))
print()
print('scripts available:')
for f in sorted(Path('scripts').glob('*.py')):
    print('  ', f.name)

In [ ]:
#@title 2. Settings and Google Drive { display-mode: "form" }
#@markdown Results go to Drive so a session disconnect costs time, not work.
USE_DRIVE = True  #@param {type:"boolean"}
#@markdown Seeds per mode. The published run used 6. Fewer finishes sooner but
#@markdown a difference of 0.01 macro-F1 against a seed standard deviation of
#@markdown 0.046 cannot be resolved with only two or three.
N_SEEDS = 6  #@param {type:"integer"}
EPOCHS = 30  #@param {type:"integer"}

from pathlib import Path
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE = Path('/content/drive/MyDrive/canvas_ctx_results')
else:
    SAVE = Path('/content/canvas_results')
SAVE.mkdir(parents=True, exist_ok=True)

for d in ('results', 'figures', 'reports'):
    src, dst = Path('/content/canvas') / d, SAVE / d
    dst.mkdir(parents=True, exist_ok=True)
    if src.is_symlink():
        src.unlink()
    elif src.exists():
        for f in src.rglob('*'):
            if f.is_file():
                (dst / f.relative_to(src)).parent.mkdir(parents=True, exist_ok=True)
                f.replace(dst / f.relative_to(src))
        import shutil as _s; _s.rmtree(src, ignore_errors=True)
    src.symlink_to(dst)
print('results saved to', SAVE)
print(f'{N_SEEDS} seeds x 4 modes x {EPOCHS} epochs = {N_SEEDS*4} training runs')

In [ ]:
#@title 3. Check for cached patch embeddings
# The benchmark trains a small head on FROZEN encoder embeddings. Computing
# those from whole slide images is the expensive part and is done once; the
# benchmark itself then re-trains in minutes. Without them, nothing below can
# run, and that is a data question rather than something the notebook can fix.
from pathlib import Path

EMB = Path('/content/canvas/data/interim/orion_embeddings')
found = sorted(EMB.glob('*.parquet')) if EMB.exists() else []
print(f'{len(found)} embedding shards at {EMB}')
if not found:
    print()
    print('No cached embeddings. They are not in the repository because they are')
    print('derived from whole slide images that are gigabytes each.')
    print()
    print('Two options:')
    print('  1. Upload the orion_embeddings folder from your machine to')
    print(f'     {EMB}')
    print('  2. Regenerate them: scripts/download_orion.py then')
    print('     scripts/encode_orion_patches.py. That is hours of work and')
    print('     tens of gigabytes, and only needs doing once.')
else:
    import pandas as pd
    d = pd.read_parquet(found[0])
    print('columns:', list(d.columns)[:8], '...')
    print('rows in first shard:', len(d))

In [ ]:
#@title 4. The four-way ablation: none, graph, grid2d, grid3d
# k = 0 reduces the context model EXACTLY to the per-patch baseline, so the
# comparison is an ablation of context alone rather than of two different
# architectures. Splits are at slide level: a patch-level split would place
# neighbouring patches in train and test and report memorisation.
import subprocess, sys
from pathlib import Path

if not sorted(Path('/content/canvas/data/interim/orion_embeddings').glob('*.parquet')):
    print('Skipped: no embeddings. See the previous cell.')
else:
    seeds = ' '.join(str(i) for i in range(1, N_SEEDS + 1))
    subprocess.run(
        [sys.executable, '-u', 'scripts/run_final_benchmark.py',
         '--embeddings', 'data/interim/orion_embeddings',
         '--modes', 'none', 'graph', 'grid2d', 'grid3d',
         '--seeds', *seeds.split(),
         '--epochs', str(EPOCHS), '--window', '7',
         '--outdir', 'results/real_benchmark'],
        cwd='/content/canvas')

In [ ]:
#@title 5. Recover results if the session dropped mid-run
# The benchmark writes its CSVs only after every seed and mode has finished, so
# a disconnect at 90 percent loses everything even though nearly all the work is
# done. Each completed run does log its own result line, so this reconstructs
# the table from the log and states how many of the expected runs are present.
!python scripts/salvage_benchmark_log.py results/real_benchmark.log

In [ ]:
#@title 6. Read the benchmark honestly
import pandas as pd, numpy as np
from pathlib import Path
from scipy import stats

f = Path('/content/canvas/results/real_benchmark/final_benchmark.csv')
if not f.exists():
    f = Path('/content/canvas/results/real_benchmark/final_benchmark_SALVAGED.csv')
if not f.exists():
    print('No benchmark results yet.')
else:
    d = pd.read_csv(f)
    col = 'value' if 'value' in d.columns else 'macro_f1'
    if 'metric' in d.columns:
        for met in sorted(d.metric.unique()):
            s = d[d.metric == met]
            print(f'\n=== {met} ===')
            g = s.groupby('mode')[col].agg(['count', 'mean', 'std'])
            print(g.reindex([m for m in ['none','graph','grid2d','grid3d']
                             if m in g.index]).round(4).to_string())
            base = s[s['mode'] == 'none'].set_index('seed')[col]
            for m in ['graph', 'grid2d', 'grid3d']:
                v = s[s['mode'] == m].set_index('seed')[col]
                c = base.index.intersection(v.index)
                if len(c) < 3: continue
                diff = v[c] - base[c]
                w = stats.wilcoxon(v[c], base[c])
                print(f'  {m:7} vs none: mean {diff.mean():+.4f}, '
                      f'wins {int((diff>0).sum())}/{len(c)}, p={w.pvalue:.3f}')
    print('\nA difference of about 0.01 macro-F1 sits inside a between-seed')
    print('standard deviation of roughly 0.046. Report the spread, not the mean')
    print('alone, and check the per-class recalls before trusting any of it.')

In [ ]:
#@title 7. The class collapse, which decides whether any of the above means anything
import pandas as pd
from pathlib import Path

p = Path('/content/canvas/results/real_benchmark/per_class_metrics.csv')
if not p.exists():
    print('No per-class metrics. Run the benchmark first.')
else:
    m = pd.read_csv(p)
    g = m.groupby('class').agg(mean_recall=('recall', 'mean'),
                               zero_runs=('recall', lambda s: int((s < 0.01).sum())),
                               runs=('recall', 'size'),
                               support=('support', 'mean')).round(4)
    print(g.to_string())
    dead = g[g.mean_recall < 0.02]
    print()
    print(f'{len(dead)} of {len(g)} classes are essentially never predicted.')
    print('A class with normal support and near-zero recall is not a rarity')
    print('problem. It points at the habitat definition upstream, in the')
    print('neighbourhood clustering, rather than at the classifier.')

In [ ]:
#@title 8. Download the results
from pathlib import Path
from google.colab import files
for f in ['results/real_benchmark/final_benchmark.csv',
          'results/real_benchmark/per_class_metrics.csv',
          'reports/analysis_report.html']:
    p = Path('/content/canvas') / f
    if p.exists():
        print('downloading', f)
        files.download(str(p))
print('everything is also on Drive at', SAVE)